In [3]:
!pip install sentence-transformers

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
tokenizer = model.tokenizer

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.9 MB/s eta 0:00:00


In [58]:
!pip install rank_bm25

In [59]:
import faiss
from rank_bm25 import BM25Okapi

In [7]:
# After making a basic Naive RAG now i am implementing 3 techinques of chunking (Fixed No overlap, Fixed Overlapping, Recursive)
# Then adding Evaluation Methods (Precision Recall and MRR)
# Then Adding a VectorDB and ANN searching via FAISS or Numpy array
# Adding Hybrid Retrieval BM-25 and SBERT USING BM250K LIBRARY INSTEAD OF REWRITING THE CODE
# Adding Re - Ranker (raw Bert)
# WILL ADD RAGAS

CORPUS


In [8]:
corpus_dict = {
    "photosynthesis_A": "Photosynthesis is the process by which plants convert light energy into chemical energy. It occurs primarily in the chloroplasts, specifically within structures called thylakoids. Chlorophyll, the green pigment in plants, absorbs light most efficiently in the blue and red wavelengths while reflecting green light. The overall reaction converts carbon dioxide and water into glucose and oxygen using energy from sunlight. The light-dependent reactions of photosynthesis take place across the thylakoid membrane. Water molecules are split in a process called photolysis, releasing oxygen as a byproduct. This stage produces ATP and NADPH, which are then used to power the second stage of photosynthesis. Without sufficient light intensity, the rate of these reactions slows significantly, limiting overall plant growth.",

    "photosynthesis_B": "The Calvin cycle, also called the light-independent reactions, occurs in the stroma of the chloroplast. Carbon dioxide is fixed into organic molecules using the ATP and NADPH generated earlier. This cycle does not directly require light, but depends entirely on the products of the light-dependent reactions to function. The end product of the Calvin cycle is glucose, which the plant uses for energy or converts into starch for storage. Certain plants have evolved alternative photosynthetic pathways to cope with hot, dry climates. C4 plants, such as corn and sugarcane, spatially separate the initial carbon fixation step from the Calvin cycle to minimize water loss. CAM plants, like cacti, instead separate these processes by time, opening their stomata only at night to reduce evaporation while still collecting carbon dioxide.",

    "photosynthesis_C": "Photorespiration is a wasteful side reaction that occurs when the enzyme RuBisCO binds oxygen instead of carbon dioxide, particularly under hot, dry conditions when stomata close to conserve water. This process consumes energy and releases previously fixed carbon dioxide without producing any usable sugar, effectively undoing some of the work of the Calvin cycle. C4 and CAM pathways are, in large part, evolutionary adaptations specifically aimed at minimizing photorespiration's cost. Scientists have explored genetically engineering photorespiration bypass pathways into C3 crops like rice and wheat to improve yield, since these staple crops lack the C4 or CAM adaptations that reduce this energy loss. Chloroplasts are believed to have originated through endosymbiosis, a process in which an ancestral eukaryotic cell engulfed a photosynthetic cyanobacterium roughly 1.5 billion years ago rather than digesting it, and the two entered a mutually beneficial relationship that became permanent over evolutionary time. Evidence for this theory includes the fact that chloroplasts retain their own small circular DNA genome, separate from the plant cell's nuclear DNA, and replicate independently within the cell through a process resembling bacterial binary fission. Chloroplasts also possess their own ribosomes, structurally more similar to bacterial ribosomes than to the plant cell's own cytoplasmic ribosomes. This same endosymbiotic origin story is thought to explain mitochondria as well, though from a different bacterial ancestor entirely, making endosymbiosis one of the most significant events in the evolutionary history of complex life on Earth.",

    "french_revolution_A": "The French Revolution began in 1789, driven by widespread financial crisis, food shortages, and deep resentment toward the rigid social hierarchy of the Ancien Régime. The Estates-General was convened in May 1789 for the first time in over a century, as King Louis XVI sought new taxes to address the state's crippling debt. Tensions between the three estates quickly escalated, culminating in the formation of the National Assembly by representatives of the Third Estate. The storming of the Bastille on July 14, 1789 became a defining symbol of the Revolution. The prison held few political prisoners at the time, but it represented royal tyranny and served as a source of weapons and gunpowder for the revolutionaries. News of the event spread rapidly through France, triggering uprisings in the countryside known as the Great Fear, during which peasants attacked manor houses and destroyed feudal records.",

    "french_revolution_B": "The Reign of Terror, lasting from 1793 to 1794, saw the revolutionary government under Robespierre execute thousands of perceived enemies of the Revolution using the guillotine. The Committee of Public Safety wielded near-total power during this period, justifying mass executions as necessary to protect the Revolution from internal and external threats. Robespierre himself was eventually arrested and executed in July 1794, effectively ending the Terror. Napoleon Bonaparte rose to power in the aftermath of the Revolution's instability, staging a coup in 1799 to establish the Consulate. He would later crown himself Emperor in 1804, effectively ending the revolutionary republic while retaining many of its legal and administrative reforms. The Napoleonic Code, established during his rule, standardized civil law across France and influenced legal systems throughout Europe.",

    "french_revolution_C": "The Directory governed France from 1795 to 1799, a period marked by political instability, corruption, and near-constant threat from both royalist and radical factions attempting to seize power. This five-member executive body struggled to maintain legitimacy after the excesses of the Terror had discredited more radical revolutionary governance, and it relied increasingly on the military to suppress uprisings and enforce its authority. Napoleon's own rise was made possible in large part by the Directory's chronic weakness and its growing dependence on generals to keep the government in power. The Coup of 18 Brumaire in 1799, in which Napoleon overthrew the Directory, is often treated by historians as the formal end point of the French Revolution itself. Following Napoleon's final defeat at Waterloo in 1815, European powers convened the Congress of Vienna to redraw the continent's political map and restore a balance of power disrupted by over two decades of revolutionary and Napoleonic war. The Bourbon monarchy was restored in France under Louis XVIII, though the restored government was forced to retain many of the legal and social reforms of the revolutionary period, since reversing them entirely was no longer politically feasible. The Congress deliberately prioritized conservative stability over the nationalist and liberal ideals the Revolution had unleashed across Europe, planting the seeds for further revolutionary movements later in the nineteenth century. Historians frequently debate how much of the Revolution's legacy actually survived this restoration versus how much was genuinely undone.",

    "http_A": "HTTP, or Hypertext Transfer Protocol, is the foundational protocol used for communication on the World Wide Web. It operates as a request-response protocol between clients, typically web browsers, and servers. Each HTTP request includes a method such as GET, POST, PUT, or DELETE, indicating the action the client wants the server to perform on a given resource. HTTP status codes are three-digit numbers returned by the server to indicate the outcome of a request. Codes in the 200 range indicate success, with 200 OK being the most common. The 300 range indicates redirection, the 400 range indicates a client-side error such as 404 Not Found, and the 500 range indicates a server-side error such as 500 Internal Server Error.",

    "http_B": "HTTP is a stateless protocol, meaning each request is processed independently without any memory of previous requests. To maintain state across multiple requests, such as keeping a user logged in, servers commonly use cookies, which are small pieces of data stored on the client and sent with each subsequent request. Session tokens and authentication headers serve similar purposes in modern web applications. HTTPS extends HTTP by adding a layer of encryption via TLS, short for Transport Layer Security. This encryption protects data in transit from eavesdropping and tampering, which is especially important when transmitting sensitive information like passwords or payment details. Most modern browsers now flag plain HTTP sites as \"not secure\" to encourage widespread adoption of HTTPS.",

    "http_C": "HTTP/2, standardized in 2015, introduced multiplexing, which allows multiple requests and responses to be sent concurrently over a single TCP connection rather than requiring a separate connection per request as earlier HTTP versions effectively did. This significantly reduced the latency overhead caused by opening and closing many parallel connections for a single page load. HTTP/3, the most recent major revision, goes further by replacing TCP entirely with a new transport protocol called QUIC, built on top of UDP, specifically to avoid a performance problem called head-of-line blocking that persisted even in HTTP/2's multiplexed connections. Adoption of HTTP/3 has grown steadily among major websites and content delivery networks since its standardization, though HTTP/1.1 remains in widespread use for simpler applications. A REST API, short for Representational State Transfer, is an architectural style for designing networked applications that relies heavily on standard HTTP methods and status codes to represent operations on resources identified by URLs. A well-designed REST API treats each URL as a distinct resource, using GET to retrieve it, POST to create a new one, PUT or PATCH to update it, and DELETE to remove it, mirroring HTTP's own verb semantics directly. REST APIs are expected to be stateless in the same sense HTTP itself is stateless, meaning the server does not retain session information between separate requests from the same client. This architectural alignment with HTTP's native design is precisely why REST became the dominant style for web APIs, rather than requiring an entirely separate protocol layered on top.",

    "return_policy_A": "Orders placed through our online store can be returned within 30 days of delivery for a full refund, provided the item is unused and in its original packaging. Refunds are processed to the original payment method within 5 to 7 business days after we receive the returned item. Customers are responsible for return shipping costs unless the item arrived damaged or defective. Items marked as final sale, including clearance products and gift cards, are not eligible for return or exchange under any circumstances. To initiate a return, customers must first request a return authorization number through their account dashboard before shipping the item back. Returns sent without this authorization number may be refused or significantly delayed in processing.",

    "return_policy_B": "For defective or damaged items, customers have 60 days from the delivery date to request a replacement or full refund, including return shipping costs. Photographic evidence of the damage is required before a replacement is shipped. Defective electronics may alternatively be exchanged directly at any physical retail location without needing to ship the item back first. Exchanges for a different size or color are treated as a separate transaction from returns. Customers must return the original item for a refund and place a new order for the desired size or color, since the company does not process direct one-to-one exchanges. During high-demand sale periods, this can occasionally result in the desired replacement item selling out before the new order is placed.",

    "return_policy_C": "International orders are subject to a modified return policy: customers have 45 days from delivery rather than the standard 30, to account for longer shipping transit times, but return shipping costs are the customer's responsibility in all cases regardless of the reason for return, including defective items, since the company does not currently offer prepaid international return labels. Customs duties and import taxes paid at the time of original delivery are non-refundable under any circumstances, even when the underlying item itself is successfully returned for a full product refund. International customers are strongly encouraged to contact customer support before initiating a return, since processing times for international refunds average 10 to 15 business days rather than the standard 5 to 7. A product warranty is distinct from the standard return policy and covers manufacturing defects that appear after the standard 60-day return window has already closed, typically extending coverage to one full year from the original purchase date. Unlike a standard return, a warranty claim does not offer a refund; it instead provides a free repair or a direct replacement of the defective unit once the claim is verified. Normal wear and tear, accidental damage, and unauthorized modifications to the product are explicitly excluded from warranty coverage, and customers must provide the original proof of purchase to file any warranty claim. Extended warranty plans, purchased separately at checkout, can extend this coverage up to three years but must be purchased within 30 days of the original order to remain valid.",

    "black_holes_A": "A black hole is a region of spacetime where gravity is so strong that nothing, not even light, can escape once it crosses the event horizon. Black holes form when massive stars collapse under their own gravity at the end of their life cycle, typically after a supernova explosion. The point at the center of a black hole, where density becomes theoretically infinite, is called a singularity. The event horizon marks the boundary beyond which escape from a black hole becomes impossible. Its size, known as the Schwarzschild radius, depends directly on the mass of the black hole. For a black hole with the mass of the Sun, the Schwarzschild radius would be approximately 3 kilometers, though stellar black holes typically have masses several times greater than the Sun.",

    "black_holes_B": "Supermassive black holes, containing millions or even billions of solar masses, are believed to exist at the center of most large galaxies, including our own Milky Way. The supermassive black hole at the center of the Milky Way, named Sagittarius A*, was directly imaged for the first time in 2022 using the Event Horizon Telescope. Its existence had previously been inferred through the orbital motion of nearby stars. Hawking radiation is a theoretical process through which black holes slowly lose mass and energy over immense timescales, eventually evaporating entirely. This phenomenon arises from quantum effects occurring near the event horizon. Because the rate of Hawking radiation is inversely related to a black hole's mass, smaller black holes are predicted to evaporate far faster than supermassive ones.",

    "black_holes_C": "The black hole information paradox arises from an apparent contradiction between general relativity and quantum mechanics: quantum mechanics holds that information about a physical system can never be truly destroyed, yet Hawking radiation appears to carry no information about what originally fell into the black hole, seemingly allowing that information to be lost forever once the black hole fully evaporates. This tension has driven decades of theoretical work, with proposed resolutions ranging from information being subtly encoded in the correlations between emitted Hawking radiation particles, to the existence of a remnant particle retaining the information after evaporation. The paradox remains formally unresolved, though most physicists now believe information is ultimately preserved in some form, based on more recent theoretical work involving quantum entanglement calculations. Stephen Hawking himself changed his position on this question over his career, having originally argued information was genuinely lost. Gravitational waves are ripples in spacetime itself, predicted by Einstein's general theory of relativity in 1916 but not directly detected until a full century later, in 2015, by the LIGO observatory. These waves are produced by extremely violent cosmic events, most notably the merger of two black holes or two neutron stars spiraling into one another and colliding. The first detected signal, designated GW150914, matched the theoretical waveform of two black holes merging with remarkable precision, providing strong independent confirmation that binary black hole systems genuinely exist and eventually merge. Since that first detection, dozens of additional merger events have been observed, opening an entirely new observational field now referred to as gravitational-wave astronomy.",

    "immune_system_A": "The immune system is the body's primary defense network against pathogens such as bacteria, viruses, fungi, and parasites. It is broadly divided into two major branches: the innate immune system, which provides a fast, general-purpose response present from birth, and the adaptive immune system, which develops a highly specific response over the course of an individual's life. Innate immunity includes physical barriers like skin and mucous membranes, along with immune cells that attack anything recognized as foreign without needing prior exposure to that specific threat. White blood cells, also called leukocytes, are the primary cellular agents of the immune system and circulate through blood and lymphatic tissue searching for signs of infection or damage. Neutrophils, the most abundant type, are typically first to arrive at a site of infection and work by engulfing and destroying pathogens directly. Macrophages perform a similar engulfing function but also play a crucial role in signaling other immune cells and clearing away dead cellular debris after an infection has been resolved.",

    "immune_system_B": "Antibodies are Y-shaped proteins produced by specialized white blood cells called B cells, and each antibody is precisely shaped to bind to one specific target, known as an antigen, found on the surface of a pathogen. Once bound, an antibody can neutralize a pathogen directly by blocking it from infecting cells, or it can tag the pathogen for destruction by other immune cells. B cells that have encountered a particular antigen before can persist for years or even decades as memory B cells, allowing for a dramatically faster antibody response the next time that same pathogen is encountered. Vaccines work by deliberately exposing the immune system to a harmless version or fragment of a pathogen, training the adaptive immune system to recognize it without causing the actual disease. This exposure prompts the formation of memory B cells and memory T cells specific to that pathogen, so that a real future infection can be identified and neutralized rapidly, often before the person experiences any noticeable symptoms at all. Different vaccine technologies achieve this in different ways: some use a weakened live version of the pathogen, others use an inactivated version, and newer mRNA vaccines instead instruct the body's own cells to temporarily produce a single harmless viral protein that the immune system learns to recognize.",

    "immune_system_C": "An autoimmune disease occurs when the immune system mistakenly identifies the body's own healthy cells as foreign and mounts an attack against them, essentially a case of mistaken identity at the molecular level. Rheumatoid arthritis, in which the immune system attacks joint tissue, and type 1 diabetes, in which it attacks insulin-producing cells in the pancreas, are two well-known examples. The exact triggers for most autoimmune diseases remain only partially understood, though a combination of genetic predisposition and environmental factors, such as certain infections, is generally believed to play a role in most cases. When a pathogen breaches the body's initial physical barriers, the innate immune system responds within minutes to hours through a coordinated set of mechanisms including inflammation, the recruitment of neutrophils and macrophages to the affected site, and the release of signaling proteins called cytokines that coordinate the wider immune response. If the innate response alone proves insufficient to clear the infection, dendritic cells present fragments of the pathogen to T cells, effectively bridging the innate and adaptive systems and activating a much more targeted response over the following days. This activation triggers both the antibody-producing B cell response described earlier and a separate T cell response, in which cytotoxic T cells directly destroy the body's own cells that have already been infected by the pathogen, preventing further viral replication within them.",

    "blockchain_A": "A blockchain is a distributed digital ledger that records transactions across a network of computers in a way designed to make retroactively altering past records extremely difficult without altering every subsequent block in the chain. Each block contains a set of transactions along with a cryptographic hash of the previous block, which is what links the blocks together into an unbroken, tamper-evident chain. Because copies of the ledger are distributed across many independent participants in the network rather than stored in one central location, no single party can unilaterally rewrite the transaction history. Mining is the process by which new transactions are verified and added to a blockchain, and on many blockchains it also involves solving a computationally difficult mathematical puzzle in order to earn the right to add the next block. This puzzle-solving mechanism, called proof of work, requires significant computing power and electricity, which is precisely what makes it costly and impractical for a malicious actor to rewrite the blockchain's history. Some newer blockchain networks instead use proof of stake, a different consensus mechanism in which validators are selected to add new blocks based on the amount of cryptocurrency they have locked up as collateral, rather than on raw computational power.",

    "blockchain_B": "Bitcoin, introduced in 2009 by an anonymous individual or group using the pseudonym Satoshi Nakamoto, was the first successful implementation of a blockchain-based cryptocurrency and remains the largest by total market value. Its total supply is permanently capped at 21 million coins, a limit enforced directly by the protocol's own rules rather than by any central authority's discretion. New bitcoins are gradually released into circulation as a reward to miners for successfully adding new blocks, though this reward amount is programmed to be cut in half approximately every four years, an event known as the halving. A smart contract is self-executing code deployed directly onto a blockchain that automatically carries out predefined actions once its specified conditions are met, without requiring a trusted intermediary to enforce the agreement. Ethereum, launched in 2015, was the first major blockchain platform specifically designed to support general-purpose smart contracts, rather than being narrowly focused on simple currency transactions the way Bitcoin's original design was. Smart contracts have since been used to build decentralized applications spanning lending, insurance, digital collectibles, and numerous other use cases well beyond basic peer-to-peer payments.",

    "blockchain_C": "A cryptocurrency wallet does not actually store any coins directly; instead, it stores the cryptographic keys that prove ownership of coins recorded on the blockchain itself and grant the authority to spend them. Each wallet has a public key, which functions similarly to an account number that can be safely shared with others, and a private key, which must be kept strictly secret since anyone who obtains it gains full control over the associated funds. Losing a private key with no backup means permanently losing access to the associated cryptocurrency, since there is no central authority capable of resetting it or recovering the funds on the owner's behalf. Cryptocurrency prices are notoriously volatile compared to most traditional financial assets, with double-digit percentage swings in value occurring within a single day not being unusual during periods of high market activity. This volatility stems from a combination of relatively low market liquidity compared to major traditional currencies, speculative trading behavior, and a regulatory environment that still varies significantly and continues to shift from country to country. Critics frequently point to this volatility as a major practical barrier preventing cryptocurrencies from functioning as reliable everyday currency, while proponents tend to argue that volatility will naturally decrease over time as broader adoption and market liquidity both continue to increase.",
}

In [9]:
corpus = """
Photosynthesis is the process by which plants convert light energy into chemical energy. It occurs primarily in the chloroplasts, specifically within structures called thylakoids. Chlorophyll, the green pigment in plants, absorbs light most efficiently in the blue and red wavelengths while reflecting green light. The overall reaction converts carbon dioxide and water into glucose and oxygen using energy from sunlight.
The light-dependent reactions of photosynthesis take place across the thylakoid membrane. Water molecules are split in a process called photolysis, releasing oxygen as a byproduct. This stage produces ATP and NADPH, which are then used to power the second stage of photosynthesis. Without sufficient light intensity, the rate of these reactions slows significantly, limiting overall plant growth.
The Calvin cycle, also called the light-independent reactions, occurs in the stroma of the chloroplast. Carbon dioxide is fixed into organic molecules using the ATP and NADPH generated earlier. This cycle does not directly require light, but depends entirely on the products of the light-dependent reactions to function. The end product of the Calvin cycle is glucose, which the plant uses for energy or converts into starch for storage.
Certain plants have evolved alternative photosynthetic pathways to cope with hot, dry climates. C4 plants, such as corn and sugarcane, spatially separate the initial carbon fixation step from the Calvin cycle to minimize water loss. CAM plants, like cacti, instead separate these processes by time, opening their stomata only at night to reduce evaporation while still collecting carbon dioxide.
Photorespiration is a wasteful side reaction that occurs when the enzyme RuBisCO binds oxygen instead of carbon dioxide, particularly under hot, dry conditions when stomata close to conserve water. This process consumes energy and releases previously fixed carbon dioxide without producing any usable sugar, effectively undoing some of the work of the Calvin cycle. C4 and CAM pathways are, in large part, evolutionary adaptations specifically aimed at minimizing photorespiration's cost. Scientists have explored genetically engineering photorespiration bypass pathways into C3 crops like rice and wheat to improve yield, since these staple crops lack the C4 or CAM adaptations that reduce this energy loss.
Chloroplasts are believed to have originated through endosymbiosis, a process in which an ancestral eukaryotic cell engulfed a photosynthetic cyanobacterium roughly 1.5 billion years ago rather than digesting it, and the two entered a mutually beneficial relationship that became permanent over evolutionary time. Evidence for this theory includes the fact that chloroplasts retain their own small circular DNA genome, separate from the plant cell's nuclear DNA, and replicate independently within the cell through a process resembling bacterial binary fission. Chloroplasts also possess their own ribosomes, structurally more similar to bacterial ribosomes than to the plant cell's own cytoplasmic ribosomes. This same endosymbiotic origin story is thought to explain mitochondria as well, though from a different bacterial ancestor entirely, making endosymbiosis one of the most significant events in the evolutionary history of complex life on Earth.

The French Revolution began in 1789, driven by widespread financial crisis, food shortages, and deep resentment toward the rigid social hierarchy of the Ancien Régime. The Estates-General was convened in May 1789 for the first time in over a century, as King Louis XVI sought new taxes to address the state's crippling debt. Tensions between the three estates quickly escalated, culminating in the formation of the National Assembly by representatives of the Third Estate.
The storming of the Bastille on July 14, 1789 became a defining symbol of the Revolution. The prison held few political prisoners at the time, but it represented royal tyranny and served as a source of weapons and gunpowder for the revolutionaries. News of the event spread rapidly through France, triggering uprisings in the countryside known as the Great Fear, during which peasants attacked manor houses and destroyed feudal records.
The Reign of Terror, lasting from 1793 to 1794, saw the revolutionary government under Robespierre execute thousands of perceived enemies of the Revolution using the guillotine. The Committee of Public Safety wielded near-total power during this period, justifying mass executions as necessary to protect the Revolution from internal and external threats. Robespierre himself was eventually arrested and executed in July 1794, effectively ending the Terror.
Napoleon Bonaparte rose to power in the aftermath of the Revolution's instability, staging a coup in 1799 to establish the Consulate. He would later crown himself Emperor in 1804, effectively ending the revolutionary republic while retaining many of its legal and administrative reforms. The Napoleonic Code, established during his rule, standardized civil law across France and influenced legal systems throughout Europe.
The Directory governed France from 1795 to 1799, a period marked by political instability, corruption, and near-constant threat from both royalist and radical factions attempting to seize power. This five-member executive body struggled to maintain legitimacy after the excesses of the Terror had discredited more radical revolutionary governance, and it relied increasingly on the military to suppress uprisings and enforce its authority. Napoleon's own rise was made possible in large part by the Directory's chronic weakness and its growing dependence on generals to keep the government in power. The Coup of 18 Brumaire in 1799, in which Napoleon overthrew the Directory, is often treated by historians as the formal end point of the French Revolution itself.
Following Napoleon's final defeat at Waterloo in 1815, European powers convened the Congress of Vienna to redraw the continent's political map and restore a balance of power disrupted by over two decades of revolutionary and Napoleonic war. The Bourbon monarchy was restored in France under Louis XVIII, though the restored government was forced to retain many of the legal and social reforms of the revolutionary period, since reversing them entirely was no longer politically feasible. The Congress deliberately prioritized conservative stability over the nationalist and liberal ideals the Revolution had unleashed across Europe, planting the seeds for further revolutionary movements later in the nineteenth century. Historians frequently debate how much of the Revolution's legacy actually survived this restoration versus how much was genuinely undone.

HTTP, or Hypertext Transfer Protocol, is the foundational protocol used for communication on the World Wide Web. It operates as a request-response protocol between clients, typically web browsers, and servers. Each HTTP request includes a method such as GET, POST, PUT, or DELETE, indicating the action the client wants the server to perform on a given resource.
HTTP status codes are three-digit numbers returned by the server to indicate the outcome of a request. Codes in the 200 range indicate success, with 200 OK being the most common. The 300 range indicates redirection, the 400 range indicates a client-side error such as 404 Not Found, and the 500 range indicates a server-side error such as 500 Internal Server Error.
HTTP is a stateless protocol, meaning each request is processed independently without any memory of previous requests. To maintain state across multiple requests, such as keeping a user logged in, servers commonly use cookies, which are small pieces of data stored on the client and sent with each subsequent request. Session tokens and authentication headers serve similar purposes in modern web applications.
HTTPS extends HTTP by adding a layer of encryption via TLS, short for Transport Layer Security. This encryption protects data in transit from eavesdropping and tampering, which is especially important when transmitting sensitive information like passwords or payment details. Most modern browsers now flag plain HTTP sites as "not secure" to encourage widespread adoption of HTTPS.
HTTP/2, standardized in 2015, introduced multiplexing, which allows multiple requests and responses to be sent concurrently over a single TCP connection rather than requiring a separate connection per request as earlier HTTP versions effectively did. This significantly reduced the latency overhead caused by opening and closing many parallel connections for a single page load. HTTP/3, the most recent major revision, goes further by replacing TCP entirely with a new transport protocol called QUIC, built on top of UDP, specifically to avoid a performance problem called head-of-line blocking that persisted even in HTTP/2's multiplexed connections. Adoption of HTTP/3 has grown steadily among major websites and content delivery networks since its standardization, though HTTP/1.1 remains in widespread use for simpler applications.
A REST API, short for Representational State Transfer, is an architectural style for designing networked applications that relies heavily on standard HTTP methods and status codes to represent operations on resources identified by URLs. A well-designed REST API treats each URL as a distinct resource, using GET to retrieve it, POST to create a new one, PUT or PATCH to update it, and DELETE to remove it, mirroring HTTP's own verb semantics directly. REST APIs are expected to be stateless in the same sense HTTP itself is stateless, meaning the server does not retain session information between separate requests from the same client. This architectural alignment with HTTP's native design is precisely why REST became the dominant style for web APIs, rather than requiring an entirely separate protocol layered on top.

Orders placed through our online store can be returned within 30 days of delivery for a full refund, provided the item is unused and in its original packaging. Refunds are processed to the original payment method within 5 to 7 business days after we receive the returned item. Customers are responsible for return shipping costs unless the item arrived damaged or defective.
Items marked as final sale, including clearance products and gift cards, are not eligible for return or exchange under any circumstances. To initiate a return, customers must first request a return authorization number through their account dashboard before shipping the item back. Returns sent without this authorization number may be refused or significantly delayed in processing.
For defective or damaged items, customers have 60 days from the delivery date to request a replacement or full refund, including return shipping costs. Photographic evidence of the damage is required before a replacement is shipped. Defective electronics may alternatively be exchanged directly at any physical retail location without needing to ship the item back first.
Exchanges for a different size or color are treated as a separate transaction from returns. Customers must return the original item for a refund and place a new order for the desired size or color, since the company does not process direct one-to-one exchanges. During high-demand sale periods, this can occasionally result in the desired replacement item selling out before the new order is placed.
International orders are subject to a modified return policy: customers have 45 days from delivery rather than the standard 30, to account for longer shipping transit times, but return shipping costs are the customer's responsibility in all cases regardless of the reason for return, including defective items, since the company does not currently offer prepaid international return labels. Customs duties and import taxes paid at the time of original delivery are non-refundable under any circumstances, even when the underlying item itself is successfully returned for a full product refund. International customers are strongly encouraged to contact customer support before initiating a return, since processing times for international refunds average 10 to 15 business days rather than the standard 5 to 7.
A product warranty is distinct from the standard return policy and covers manufacturing defects that appear after the standard 60-day return window has already closed, typically extending coverage to one full year from the original purchase date. Unlike a standard return, a warranty claim does not offer a refund; it instead provides a free repair or a direct replacement of the defective unit once the claim is verified. Normal wear and tear, accidental damage, and unauthorized modifications to the product are explicitly excluded from warranty coverage, and customers must provide the original proof of purchase to file any warranty claim. Extended warranty plans, purchased separately at checkout, can extend this coverage up to three years but must be purchased within 30 days of the original order to remain valid.

A black hole is a region of spacetime where gravity is so strong that nothing, not even light, can escape once it crosses the event horizon. Black holes form when massive stars collapse under their own gravity at the end of their life cycle, typically after a supernova explosion. The point at the center of a black hole, where density becomes theoretically infinite, is called a singularity.
The event horizon marks the boundary beyond which escape from a black hole becomes impossible. Its size, known as the Schwarzschild radius, depends directly on the mass of the black hole. For a black hole with the mass of the Sun, the Schwarzschild radius would be approximately 3 kilometers, though stellar black holes typically have masses several times greater than the Sun.
Supermassive black holes, containing millions or even billions of solar masses, are believed to exist at the center of most large galaxies, including our own Milky Way. The supermassive black hole at the center of the Milky Way, named Sagittarius A*, was directly imaged for the first time in 2022 using the Event Horizon Telescope. Its existence had previously been inferred through the orbital motion of nearby stars.
Hawking radiation is a theoretical process through which black holes slowly lose mass and energy over immense timescales, eventually evaporating entirely. This phenomenon arises from quantum effects occurring near the event horizon. Because the rate of Hawking radiation is inversely related to a black hole's mass, smaller black holes are predicted to evaporate far faster than supermassive ones.
The black hole information paradox arises from an apparent contradiction between general relativity and quantum mechanics: quantum mechanics holds that information about a physical system can never be truly destroyed, yet Hawking radiation appears to carry no information about what originally fell into the black hole, seemingly allowing that information to be lost forever once the black hole fully evaporates. This tension has driven decades of theoretical work, with proposed resolutions ranging from information being subtly encoded in the correlations between emitted Hawking radiation particles, to the existence of a remnant particle retaining the information after evaporation. The paradox remains formally unresolved, though most physicists now believe information is ultimately preserved in some form, based on more recent theoretical work involving quantum entanglement calculations. Stephen Hawking himself changed his position on this question over his career, having originally argued information was genuinely lost.
Gravitational waves are ripples in spacetime itself, predicted by Einstein's general theory of relativity in 1916 but not directly detected until a full century later, in 2015, by the LIGO observatory. These waves are produced by extremely violent cosmic events, most notably the merger of two black holes or two neutron stars spiraling into one another and colliding. The first detected signal, designated GW150914, matched the theoretical waveform of two black holes merging with remarkable precision, providing strong independent confirmation that binary black hole systems genuinely exist and eventually merge. Since that first detection, dozens of additional merger events have been observed, opening an entirely new observational field now referred to as gravitational-wave astronomy.

The immune system is the body's primary defense network against pathogens such as bacteria, viruses, fungi, and parasites. It is broadly divided into two major branches: the innate immune system, which provides a fast, general-purpose response present from birth, and the adaptive immune system, which develops a highly specific response over the course of an individual's life. Innate immunity includes physical barriers like skin and mucous membranes, along with immune cells that attack anything recognized as foreign without needing prior exposure to that specific threat.
White blood cells, also called leukocytes, are the primary cellular agents of the immune system and circulate through blood and lymphatic tissue searching for signs of infection or damage. Neutrophils, the most abundant type, are typically first to arrive at a site of infection and work by engulfing and destroying pathogens directly. Macrophages perform a similar engulfing function but also play a crucial role in signaling other immune cells and clearing away dead cellular debris after an infection has been resolved.
Antibodies are Y-shaped proteins produced by specialized white blood cells called B cells, and each antibody is precisely shaped to bind to one specific target, known as an antigen, found on the surface of a pathogen. Once bound, an antibody can neutralize a pathogen directly by blocking it from infecting cells, or it can tag the pathogen for destruction by other immune cells. B cells that have encountered a particular antigen before can persist for years or even decades as memory B cells, allowing for a dramatically faster antibody response the next time that same pathogen is encountered.
Vaccines work by deliberately exposing the immune system to a harmless version or fragment of a pathogen, training the adaptive immune system to recognize it without causing the actual disease. This exposure prompts the formation of memory B cells and memory T cells specific to that pathogen, so that a real future infection can be identified and neutralized rapidly, often before the person experiences any noticeable symptoms at all. Different vaccine technologies achieve this in different ways: some use a weakened live version of the pathogen, others use an inactivated version, and newer mRNA vaccines instead instruct the body's own cells to temporarily produce a single harmless viral protein that the immune system learns to recognize.
An autoimmune disease occurs when the immune system mistakenly identifies the body's own healthy cells as foreign and mounts an attack against them, essentially a case of mistaken identity at the molecular level. Rheumatoid arthritis, in which the immune system attacks joint tissue, and type 1 diabetes, in which it attacks insulin-producing cells in the pancreas, are two well-known examples. The exact triggers for most autoimmune diseases remain only partially understood, though a combination of genetic predisposition and environmental factors, such as certain infections, is generally believed to play a role in most cases.
When a pathogen breaches the body's initial physical barriers, the innate immune system responds within minutes to hours through a coordinated set of mechanisms including inflammation, the recruitment of neutrophils and macrophages to the affected site, and the release of signaling proteins called cytokines that coordinate the wider immune response. If the innate response alone proves insufficient to clear the infection, dendritic cells present fragments of the pathogen to T cells, effectively bridging the innate and adaptive systems and activating a much more targeted response over the following days. This activation triggers both the antibody-producing B cell response described earlier and a separate T cell response, in which cytotoxic T cells directly destroy the body's own cells that have already been infected by the pathogen, preventing further viral replication within them.

A blockchain is a distributed digital ledger that records transactions across a network of computers in a way designed to make retroactively altering past records extremely difficult without altering every subsequent block in the chain. Each block contains a set of transactions along with a cryptographic hash of the previous block, which is what links the blocks together into an unbroken, tamper-evident chain. Because copies of the ledger are distributed across many independent participants in the network rather than stored in one central location, no single party can unilaterally rewrite the transaction history.
Mining is the process by which new transactions are verified and added to a blockchain, and on many blockchains it also involves solving a computationally difficult mathematical puzzle in order to earn the right to add the next block. This puzzle-solving mechanism, called proof of work, requires significant computing power and electricity, which is precisely what makes it costly and impractical for a malicious actor to rewrite the blockchain's history. Some newer blockchain networks instead use proof of stake, a different consensus mechanism in which validators are selected to add new blocks based on the amount of cryptocurrency they have locked up as collateral, rather than on raw computational power.
Bitcoin, introduced in 2009 by an anonymous individual or group using the pseudonym Satoshi Nakamoto, was the first successful implementation of a blockchain-based cryptocurrency and remains the largest by total market value. Its total supply is permanently capped at 21 million coins, a limit enforced directly by the protocol's own rules rather than by any central authority's discretion. New bitcoins are gradually released into circulation as a reward to miners for successfully adding new blocks, though this reward amount is programmed to be cut in half approximately every four years, an event known as the halving.
A smart contract is self-executing code deployed directly onto a blockchain that automatically carries out predefined actions once its specified conditions are met, without requiring a trusted intermediary to enforce the agreement. Ethereum, launched in 2015, was the first major blockchain platform specifically designed to support general-purpose smart contracts, rather than being narrowly focused on simple currency transactions the way Bitcoin's original design was. Smart contracts have since been used to build decentralized applications spanning lending, insurance, digital collectibles, and numerous other use cases well beyond basic peer-to-peer payments.
A cryptocurrency wallet does not actually store any coins directly; instead, it stores the cryptographic keys that prove ownership of coins recorded on the blockchain itself and grant the authority to spend them. Each wallet has a public key, which functions similarly to an account number that can be safely shared with others, and a private key, which must be kept strictly secret since anyone who obtains it gains full control over the associated funds. Losing a private key with no backup means permanently losing access to the associated cryptocurrency, since there is no central authority capable of resetting it or recovering the funds on the owner's behalf.
Cryptocurrency prices are notoriously volatile compared to most traditional financial assets, with double-digit percentage swings in value occurring within a single day not being unusual during periods of high market activity. This volatility stems from a combination of relatively low market liquidity compared to major traditional currencies, speculative trading behavior, and a regulatory environment that still varies significantly and continues to shift from country to country. Critics frequently point to this volatility as a major practical barrier preventing cryptocurrencies from functioning as reliable everyday currency, while proponents tend to argue that volatility will naturally decrease over time as broader adoption and market liquidity both continue to increase.
"""
para_ids = [
    # Photosynthesis (0-5)
    "photosynthesis_1",
    "photosynthesis_2",
    "photosynthesis_3",
    "photosynthesis_4",
    "photosynthesis_5",   # photorespiration
    "photosynthesis_6",   # endosymbiotic origin — long paragraph

    # French Revolution (6-11)
    "french_revolution_1",
    "french_revolution_2",
    "french_revolution_3",
    "french_revolution_4",
    "french_revolution_5",   # the Directory
    "french_revolution_6",   # Congress of Vienna — long paragraph

    # HTTP / HTTPS (12-17)
    "http_1",
    "http_2",
    "http_3",
    "http_4",
    "http_5",   # HTTP/2 and HTTP/3
    "http_6",   # REST APIs — long paragraph

    # Return policy (18-23)
    "return_policy_1",
    "return_policy_2",
    "return_policy_3",
    "return_policy_4",
    "return_policy_5",   # international returns
    "return_policy_6",   # warranty vs return — long paragraph

    # Black holes (24-29)
    "black_holes_1",
    "black_holes_2",
    "black_holes_3",
    "black_holes_4",
    "black_holes_5",   # information paradox — long paragraph
    "black_holes_6",   # gravitational waves / LIGO

    # Immune system (30-35)
    "immune_system_1",
    "immune_system_2",
    "immune_system_3",
    "immune_system_4",
    "immune_system_5",   # autoimmune disease
    "immune_system_6",   # innate-to-adaptive bridge — long paragraph

    # Blockchain / cryptocurrency (36-41)
    "blockchain_1",
    "blockchain_2",   # mining / consensus — long paragraph
    "blockchain_3",
    "blockchain_4",
    "blockchain_5",
    "blockchain_6",
]

In [10]:
# Instead of Corpus being a strings and then converting to list of strings and removing the separators and then also creating the para/chunk_ids as a list then
# creating a doc_id_to_doc dict i can directly accept a dict of the raw doc doc_dict[doc_id(name)] = Actual document

OLD CODE REUSED


In [11]:
# MY OLD CODE HAD MULTIPLE BUGS OF FALSY CONDITION STATEMENTS
# def input_processing(corpus_str = None, para_id_list= None, corpus_dict = None, isDict = False, isParaSplit = False):
#   if not corpus_str and not corpus_dict : raise Exception("No Corpus Given")
#   if isDict :
#     return list(corpus_dict.keys()) , list(corpus_dict.values())
#   else :
#     if isParaSplit  : pre_pro_docs = corpus_str.split("\n\n")
#     else : pre_pro_docs = corpus_str.split('\n')
#     output_corpus = [doc for doc in pre_pro_docs if doc.strip()]
#     print("Length of output corpus list ",len(output_corpus))

#     return para_id_list , output_corpus

#   return None , None # DEAD CODE / UNREACHABLE





In [50]:
def input_processing(corpus_str=None, para_id_list=None, corpus_dict=None, isDict=False, isParaSplit=False):
    if corpus_str is None and corpus_dict is None:
        raise Exception("No Corpus Given")

    if isDict:
        if corpus_dict is None:
            raise Exception("isDict=True but no corpus_dict given")
        return list(corpus_dict.keys()), list(corpus_dict.values())

    if isParaSplit:
        pre_pro_docs = corpus_str.split("\n\n")
    else:
        pre_pro_docs = corpus_str.split('\n')

    # filter empty/whitespace-only lines FIRST — this removes the blank-line
    # artifacts between topics before any length comparison happens
    output_corpus = [doc.strip() for doc in pre_pro_docs if doc.strip()]

    if para_id_list is not None:
        if len(para_id_list) != len(output_corpus):
            raise ValueError(
                f"Length mismatch AFTER filtering: {len(para_id_list)} ids vs "
                f"{len(output_corpus)} non-empty docs — check your split logic "
                f"or para_ids ordering"
            )

    print("Length of output corpus list ", len(output_corpus))
    return para_id_list, output_corpus

In [51]:
# TEST FUNC -1
# print(input_processing(corpus_str=corpus , para_id_list=para_ids))

In [52]:
# MY OLD CODE ALSO FALSY CONDITION CHECKS
# def get_corpus_dict(corpus_list = None , para_ids_list = None, isDict = False):
#   if not corpus_list or not para_ids_list : raise Exception("Error")
#   if isDict : raise Exception("Already Having a corpus_dict")
#   corpus_dict = {}
#   return dict(zip(para_ids_list,corpus_list))

In [53]:
# NEW AI GENERATED CODE
def get_corpus_dict(corpus_list=None, para_ids_list=None, isDict=False):
    if not corpus_list or not para_ids_list:
        raise Exception("Error: corpus_list or para_ids_list is empty/None")
    if isDict:
        raise Exception("Already Having a corpus_dict")
    if len(corpus_list) != len(para_ids_list):
        raise ValueError(
            f"Length mismatch: {len(para_ids_list)} ids vs {len(corpus_list)} docs"
        )
    if len(set(para_ids_list)) != len(para_ids_list):
        raise ValueError("Duplicate doc_ids found in para_ids_list")

    return dict(zip(para_ids_list, corpus_list))

In [55]:
# # TEST FUNC - 2
a,b = input_processing(corpus_str=corpus , para_id_list=para_ids)
c = get_corpus_dict(corpus_list=b,para_ids_list=para_ids)
len(c)

Length of output corpus list  42


42

In [15]:
def check_corpus_length(tokenizer = None,corpus_dict = None ):
  if not tokenizer : raise Exception("No Tokenizer Given")
  if not corpus_dict : raise Exception("No Corpus Given")
  for para_id,doc in corpus_dict.items() :
    doc_tokens = len(tokenizer.encode(doc))
    print(f"The doc with doc_id {para_id} has tokens {doc_tokens}")

In [16]:
# TEST FUNC - 3
check_corpus_length(tokenizer=tokenizer,corpus_dict=corpus_dict)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (333 > 256). Running this sequence through the model will result in indexing errors


The doc with doc_id photosynthesis_A has tokens 161
The doc with doc_id photosynthesis_B has tokens 171
The doc with doc_id photosynthesis_C has tokens 333
The doc with doc_id french_revolution_A has tokens 175
The doc with doc_id french_revolution_B has tokens 155
The doc with doc_id french_revolution_C has tokens 289
The doc with doc_id http_A has tokens 152
The doc with doc_id http_B has tokens 145
The doc with doc_id http_C has tokens 318
The doc with doc_id return_policy_A has tokens 132
The doc with doc_id return_policy_B has tokens 141
The doc with doc_id return_policy_C has tokens 292
The doc with doc_id black_holes_A has tokens 155
The doc with doc_id black_holes_B has tokens 166
The doc with doc_id black_holes_C has tokens 314
The doc with doc_id immune_system_A has tokens 213
The doc with doc_id immune_system_B has tokens 244
The doc with doc_id immune_system_C has tokens 292
The doc with doc_id blockchain_A has tokens 253
The doc with doc_id blockchain_B has tokens 251
The 

In [17]:
def _corpus_size(model = None , tokenizer = None , corpus_dict = None):
  if not tokenizer : raise Exception("No Tokenizer Given")
  if not corpus_dict : raise Exception("No Corpus Given")
  if not model : raise Exception("No Model Given")
  out_range_docs = []
  for para_id,doc in corpus_dict.items() :
    doc_tokens = len(tokenizer.encode(doc))
    if doc_tokens > model[0].auto_model.config.max_position_embeddings :
      out_range_docs.append(para_id)
  return out_range_docs



In [18]:
# This is the Chunker class that I am building for my RAG pipeline
# The main purpose is to divide a large text into smaller chunks before embedding
# I am implementing 3 chunking approaches:
# 1. Fixed Size Non Overlapping
# 2. Fixed Size Overlapping
# 3. Recursive Chunking
# The chunk size is measured using the tokenizer in terms of tokens

# class chunker :

# __ini__()
# Initialize the Chunker with the tokenizer, maximum chunk size and overlap
# Tokenizer is required because I am measuring chunk size using tokens instead of characters
# max_length defines the maximum number of content tokens allowed in a chunk
# overlap defines how many tokens should be shared between consecutive chunks



In [19]:
# 254 content tokens + 1 [CLS] + 1 [SEP] = 256 total tokens

In [24]:


class Chunker:

  def __init__(self, tokenizer, max_length=254 , overlap = 32):
    if not tokenizer : raise Exception("No Tokenizer Given")
    if overlap >= max_length : raise ValueError(f"Overlap ({overlap}) cannot be greater than max_length ({max_length})")

    self.tokenizer = tokenizer
    self.max_length = max_length
    self.overlap = overlap

# Helper Functions

  def _token_len(self,text):
    return len(self.tokenizer.encode(text,add_special_tokens = False))

  def _decode_token(self,token_ids):
    return self.tokenizer.decode(token_ids)

  def _split_text(self,text,sep = "\n"):
    return text.split(sep)

  def _merge_chunks(self,chunks):
    temp_chunk = []
    merged_chunk = []
    for chunk in chunks :
      combined = " ".join(temp_chunk + [chunk])
      if self._token_len(combined) <= self.max_length :
        temp_chunk.append(chunk)
      else :
        if temp_chunk : merged_chunk.append(" ".join(temp_chunk))

        temp_chunk = [chunk]
    if temp_chunk : merged_chunk.append(" ".join(temp_chunk))
    return merged_chunk

  def _recursive_split(self,text,separators = ("\n\n\n",'\n\n','\n','\t','. ',' ')):
    if self._token_len(text) <= self.max_length : return [text]

    if not separators:
      return self.fixed_size_no_overlap(text)

    sep = separators[0]
    splits = self._split_text(text,sep)
    result = []

    for split in splits :
      if self._token_len(split) <= self.max_length :
        result.append(split)
        continue
      result.extend(self._recursive_split(split,separators[1:]))
    return result



 # Recursive Chunkers
  def recursive_chunk(self, text):
    splits = self._recursive_split(text)
    chunks = self._merge_chunks(splits)
    return chunks

 # Fixed Size Chunkers
  def fixed_size_no_overlap(self,text):
    tokens = self.tokenizer.encode(text,add_special_tokens = False)
    chunks = []
    for start in range(0, len(tokens),self.max_length):
      window = tokens[start:start + self.max_length]
      chunks.append(self.tokenizer.decode(window))
    return chunks


  def fixed_size_overlap(self,text):
    tokens = self.tokenizer.encode(text,add_special_tokens = False)
    chunks = []
    for start in range(0, len(tokens),self.max_length - self.overlap):
      window = tokens[start:start + self.max_length]
      chunks.append(self.tokenizer.decode(window))
    return chunks



  def chunk(self,text,method = "fixed_size_no_overlap"):
    if method == "fixed_size_no_overlap" : return self.fixed_size_no_overlap(text)
    elif method == "fixed_size_overlap" : return self.fixed_size_overlap(text)
    elif method == "recursive_no_overlap" : return self.recursive_chunk(text)
    else : raise Exception("Wrong Method")

In [ ]:
print(type(tokenizer))
print(tokenizer)

In [ ]:
# # TEST FUNC - 4
# chunker = Chunker(tokenizer=tokenizer)


# print("========== FIXED NO OVERLAP ==========")
# chunks = chunker.chunk(corpus, "fixed_size_no_overlap")

# for i, chunk in enumerate(chunks):
#     print(f"\nChunk {i+1} | Tokens: {chunker._token_len(chunk)}")
#     print(repr(chunk))


# print("\n========== FIXED WITH OVERLAP ==========")
# chunks = chunker.chunk(corpus, "fixed_size_overlap")

# for i, chunk in enumerate(chunks):
#     print(f"\nChunk {i+1} | Tokens: {chunker._token_len(chunk)}")
#     print(repr(chunk))


# print("\n========== RECURSIVE NO OVERLAP ==========")
# chunks = chunker.chunk(corpus, "recursive_no_overlap")

# for i, chunk in enumerate(chunks):
#     print(f"\nChunk {i+1} | Tokens: {chunker._token_len(chunk)}")
#     print(repr(chunk))

In [31]:
# MY OLD CODE STILL SUFFERS FROM DOWNSTREAM FALSY CONDITIONS
# def build_full_dict_per_doc_id(chunker,text,doc_id,method = 'fixed_size_no_overlap'):
#   if not chunker : raise Exception("No chunks were given")
#   if not doc_id : raise Exception("No doc_id was given")

#   chunks_per_doc = chunker.chunk(text,method)
#   result = list()

#   for i,chunk_text in enumerate(chunks_per_doc):
#     result.append({ "chunk_text": chunk_text,
#             "chunk_id": f"{doc_id}_{method}_{i}",
#             "doc_id": doc_id,
#             "chunk_index": i,
#             "chunking_method": method,
#             "token_count": chunker._token_len(chunk_text)})

#   return result





In [43]:
# NEW AI GENERATED CODE
def build_full_dict_per_doc_id(chunker, text, doc_id, method='fixed_size_no_overlap'):
    if not chunker:
        raise Exception("No chunker was given")
    if not doc_id:
        raise Exception("No doc_id was given")

    chunks_per_doc = chunker.chunk(text, method)
    result = list()

    for i, chunk_text in enumerate(chunks_per_doc):
        result.append({
            "chunk_text": chunk_text,
            "chunk_id": f"{doc_id}_{method}_{i}",
            "doc_id": doc_id,
            "chunk_index": i,
            "chunking_method": method,
            "token_count": chunker._token_len(chunk_text),  # STILL ACCESSING THE SEMI PRIVATE METHOD IN CHUNKER CLASS IDTS ANY CODE WILL BREAK BECAUSE OF THIS
        })

    return result

In [33]:
# TEST FUNC - 5
# c is dict(doc_id : full text )
# print(build_full_dict_per_doc_id(chunker,c['black_holes_1'],'black_hole_1'))
#def get_corpus_dict(corpus_list = None , para_ids_list = None, isDict = False):
#def input_processing(corpus_str = None, para_id_list= None, corpus_dict = None, isDict = False, isParaSplit = False):

In [34]:
# OLD CODE SAME CONDITION PROBLEM + CHECKING CHUNKER CLASS INSTEAD OF THE CHUNKER OBJECT AS THE PARAMETER
# def process_all_docs_metadata(chunker = None ,corpus_str = None,para_ids_list = None ,corpus_dict = None, isDict = False, isParaSplit = False):
#   if not Chunker : raise Exception("No Chunker Given ")
#   para_ids,output_corpus = input_processing(corpus_str=corpus_str,para_id_list=para_ids_list,corpus_dict=corpus_dict,isDict=isDict,isParaSplit=isParaSplit)
#   output_corpus_dict = get_corpus_dict(corpus_list=output_corpus,para_ids_list=para_ids)
#   all_records = list()
#   for key,val in output_corpus_dict.items():
#     all_records.extend(build_full_dict_per_doc_id(chunker,val,key))
#   return all_records


In [42]:
# NEW AI GENERATED CODE
def process_all_docs_metadata(chunker=None, corpus_str=None, para_ids_list=None, corpus_dict=None,
                               isDict=False, isParaSplit=False, method='fixed_size_no_overlap'):
    if not chunker:  # was checking Chunker (the class) instead of chunker (the param)
        raise Exception("No Chunker Given")

    para_ids, output_corpus = input_processing(
        corpus_str=corpus_str,
        para_id_list=para_ids_list,
        corpus_dict=corpus_dict,
        isDict=isDict,
        isParaSplit=isParaSplit
    )
    output_corpus_dict = get_corpus_dict(corpus_list=output_corpus, para_ids_list=para_ids)

    all_records = list()
    for key, val in output_corpus_dict.items():
        all_records.extend(build_full_dict_per_doc_id(chunker, val, key, method=method))

    return all_records

In [60]:
c
x = process_all_docs_metadata(chunker,corpus,para_ids)
len(x)

Length of output corpus list  42


42

Indexing (Dense + Sparse)
USING BOTH RETRIEVAL IN PARALLEL (HYBRID RETRIEVAL WITH RRF) HELPS WITH RECALL AS ANY QUERY CAN FAULT ON EITHER EXACT MATCHING ERROR OR PARAPHRASING / SYNONYMY ERROR BOTH ERRORS ARE MUTUALLY EXCLUSIVE IN NATURE



In [ ]:
# CANT USE SAME TOKENIZER OF BM25 AND SBERT AS BM25 WORKS ON WORD BASIS STATS
# TOKENIZE TEXT FOR BM25 BASIS OF SPACE
from rank_bm25 import BM25Okapi
# shape (num_chunks, dim) — aligned by index with `records`
def build_dense_index(model = None,records = None):
  if model is None or records is None : raise Exception("No Model or Records Given")
  texts = [r['chunk_text'] for r in records]
  embeds = model.encode(texts,convert_to_numpy= True ,normalize_embeddings=True , show_progress_bar= True) # AI DIDNT NORMALIZE THE VECTORS
  return embeds # DIMS(NO OF CHUNKS , DIMS OF MODEL)


# This func is building a somewhat inverted and forward index combine index for all the tokens in corpus and bm25 naturally leans more towards scoring based on cumulative
# addition of scores(query_term,corpus)
# No vectorization is needed
def build_sparse_index(records = None):
  if records is None : raise Exception("No Records Given")
  texts = [r['chunk_text'] for r in records ]
  tokenized = [text.lower().split() for text in texts]
  return BM25Okapi(tokenized)



In [67]:
# Directly call the dense search with raw query text and encode([query]) to get dim(1,dims of model)
def dense_search(query = None, embeddings = None, model = None, top_n=10):
    if query is None or embeddings is None or model is None:
        raise Exception("query, embeddings, or model is None")
    query_emb = model.encode([query], normalize_embeddings=True)
    scores = (query_emb @ embeddings.T).flatten()  # converts a matrix 2d list into 1d list of dims num_chunks
    top_indices = np.argsort(-scores)[:top_n]
    return [(idx, scores[idx]) for idx in top_indices]



def sparse_search(query = None ,bm25_idx = None , top_n=10):
    if query is None or bm25_idx is None:
        raise Exception("query or bm25_idx is None")
    query_tokens = query.lower().split()
    scores = bm25_idx.get_scores(query_tokens)
    top_indices = np.argsort(-scores)[:top_n]
    return [(idx, scores[idx]) for idx in top_indices]

In [69]:
# USING RRF (RECIPROCAL RANK FUSION) AS IT IS BLIND TO ACTUAL SCORES NO NORMALIZATION NEEDED
# ONLY DEPENDS ON RANK OF THE DOC
# ranked_lists: list of [(idx, score), ...] lists, one per retrieval method
def reciprocal_rank_fusion(ranked_lists, k=60):
    rrf_scores = {}
    for ranked_list in ranked_lists:
        for rank, (idx, _) in enumerate(ranked_list, start=1):
            rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank)
    fused = sorted(rrf_scores.items(), key=lambda x: -x[1])
    return fused  # [(idx, rrf_score), ...]

In [76]:
# NOW CREATING A FUNC THAT COMBINES BOTH THE RANKED LISTS AND RRF TOGETHER
# THIS IS OLD CODE WITH NO RERANKER IN IT WILL WRITE NEW CODE WITH RERANKER AFTER FUSED ONE AND RERANKER WILL BE OPTIONAL SO A BOOLEAN FLAG IN PARAMETER LIST
# def retrieve(query, strategy_data, top_n=10, final_k=5):
#     dense_results = dense_search(query, strategy_data["dense_embeddings"], model, top_n)
#     sparse_results = sparse_search(query, strategy_data["bm25_index"], top_n)
#     fused = reciprocal_rank_fusion([dense_results, sparse_results])[:final_k]
#     return [strategy_data["records"][idx] for idx, _ in fused]

In [75]:
# # TEST FUNC - 6
# query = "when did french revolution happened"

# # Define the chunking method
# chunking_method = 'recursive_no_overlap'

# # Process all documents to get records for the chosen chunking method
# records = process_all_docs_metadata(chunker, corpus, para_ids, method=chunking_method)

# # Build dense index
# dense_embeddings = build_dense_index(model=model, records=records)

# # Build sparse index
# bm25_index = build_sparse_index(records=records)

# # Create the indices dictionary
# indices = {
#     chunking_method: {
#         "records": records,
#         "dense_embeddings": dense_embeddings,
#         "bm25_index": bm25_index
#     }
# }

# for method, strategy_data in indices.items():
#     print(f"\n=== {method} ===")
#     results = retrieve(query, strategy_data)
#     for r in results:
#         print(f"[{r['doc_id']}] {r['chunk_text'][:80]}...")

Length of output corpus list  42


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


=== recursive_no_overlap ===
[french_revolution_1] The French Revolution began in 1789, driven by widespread financial crisis, food...
[french_revolution_5] The Directory governed France from 1795 to 1799, a period marked by political in...
[french_revolution_3] The Reign of Terror, lasting from 1793 to 1794, saw the revolutionary government...
[french_revolution_6] Following Napoleon's final defeat at Waterloo in 1815, European powers convened ...
[french_revolution_2] The storming of the Bastille on July 14, 1789 became a defining symbol of the Re...


In [77]:
# NEED TO IMPORT CROSS ENCODER AND A NEW MODEL FOR RERANKER
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [78]:
# This rerank func works only on the top_n fused rank list and re embeds , NSP STYLE LOSS FUNC TO RE RANK THEN : HELPS IN PRECISION IMPROVEMENT
# candidates: list of record dicts (from your metadata), already RRF-fused"
def rerank(query, candidates, top_k=5):

    pairs = [(query, c["chunk_text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)   # one relevance score per pair
    reranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return [c for c, s in reranked[:top_k]]

In [79]:
# NEW RETRIEVE FUNC THAT GIVES CHOICE OF USING RERANKER OR NOT
def retrieve(query, strategy_data, top_n=20, final_k=5, use_reranker=True):
    dense_results = dense_search(query, strategy_data["dense_embeddings"], model, top_n)
    sparse_results = sparse_search(query, strategy_data["bm25_index"], top_n)
    fused = reciprocal_rank_fusion([dense_results, sparse_results])[:top_n]  # keep a wider shortlist here
    candidates = [strategy_data["records"][idx] for idx, _ in fused]

    if use_reranker:
        return rerank(query, candidates, top_k=final_k)
    else:
        return candidates[:final_k]

BUILDING DENSE AND SPARSE INDICES ONCE WITH THE GIVEN CHUNKING_METHOD

In [80]:
# Define the chunking method
chunking_method = 'recursive_no_overlap'

# Process all documents to get records for the chosen chunking method
records = process_all_docs_metadata(chunker, corpus, para_ids, method=chunking_method)

# Build dense index
dense_embeddings = build_dense_index(model=model, records=records)

# Build sparse index
bm25_index = build_sparse_index(records=records)


Length of output corpus list  42


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [81]:
# TEST FUNC - 7
query = "when did french revolution happened"


# Create the indices dictionary
indices = {
    chunking_method: {
        "records": records,
        "dense_embeddings": dense_embeddings,
        "bm25_index": bm25_index
    }
}

for method, strategy_data in indices.items():
    print(f"\n=== {method} ===")
    results = retrieve(query, strategy_data)
    for r in results:
        print(f"[{r['doc_id']}] {r['chunk_text'][:80]}...")


=== recursive_no_overlap ===
[french_revolution_1] The French Revolution began in 1789, driven by widespread financial crisis, food...
[french_revolution_2] The storming of the Bastille on July 14, 1789 became a defining symbol of the Re...
[french_revolution_5] The Directory governed France from 1795 to 1799, a period marked by political in...
[french_revolution_6] Following Napoleon's final defeat at Waterloo in 1815, European powers convened ...
[french_revolution_4] Napoleon Bonaparte rose to power in the aftermath of the Revolution's instabilit...


PROMPT BUILDING AND GENERATION STAGE


In [86]:
def build_prompt(raw_query,results):
  context = "\n\n".join(r['chunk_text'] for r in results)

  prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so — do not use outside knowledge.
Context:
{context}
Question: {raw_query}
Answer:"""

  return prompt

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="API_KEY"
)

def generate_answer(prompt, client, model_name="openai/gpt-oss-20b"):
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [91]:
# query = "When did the French Revolution happen?"

# # Assuming 'recursive_no_overlap' is the desired chunking method for retrieval
# strategy_data = indices[chunking_method]

# # Retrieve documents using the defined retrieve function (with reranker by default)
# retrieved_results = retrieve(query, strategy_data, top_n=20, final_k=5, use_reranker=True)

# # Print the retrieved chunk texts for verification
# print("\n--- Retrieved Results ---")
# for i, r in enumerate(retrieved_results):
#     print(f"Result {i+1}: Doc ID: {r['doc_id']}, Chunk ID: {r['chunk_id']}\n{r['chunk_text'][:200]}...\n")

# # Build the prompt for the language model
# prompt = build_prompt(query, retrieved_results)
# print("\n--- Generated Prompt ---")
# print(prompt)

# # Generate the answer using the language model
# final_answer = generate_answer(prompt, client)
# print("\n--- Final Answer ---")
# print(final_answer)

In [88]:
# This func is ai generated lazy to integrate the pipeline and some bugs were coming
# This function encapsulates the entire Retrieval Augmented Generation (RAG) pipeline.
# It takes a raw query and uses pre-built dense and sparse indices (embeddings and BM25) to retrieve relevant documents.
# The retrieved documents are then optionally reranked, a prompt is constructed, and finally, an answer is generated by a language model.
# Combines retrieval, reranking, prompt building, and answer generation.
def RAG_Complete(
    raw_query: str,
    dense_embeddings: np.ndarray,
    bm25_index: BM25Okapi,
    records: list,
    model: SentenceTransformer,
    client: OpenAI,
    k: int = 5, # Corresponds to final_k in retrieve (number of final documents)
    use_reranker: bool = True,
    top_n_retrieval: int = 20 # Corresponds to top_n in retrieve (number of initial candidates)
) -> str:


    # Assemble strategy_data for the retrieve function
    strategy_data = {
        "records": records,
        "dense_embeddings": dense_embeddings,
        "bm25_index": bm25_index
    }

    # Retrieve documents
    retrieved_results = retrieve(
        raw_query,
        strategy_data,
        top_n=top_n_retrieval,
        final_k=k,
        use_reranker=use_reranker
    )

    # Build the prompt
    prompt = build_prompt(raw_query, retrieved_results)

    # Generate the answer
    final_answer = generate_answer(prompt, client)

    return final_answer

COMPLETE RAG PIPELINE UNDER ONE FUNCTION

In [90]:
sample_query_rag = "What are the main characteristics of a black hole and what is photosynthesis?"

# Call the RAG_Complete function using the pre-built indices and models
final_rag_answer_full = RAG_Complete(
    raw_query=sample_query_rag,
    dense_embeddings=dense_embeddings,
    bm25_index=bm25_index,
    records=records,
    model=model,
    client=client,
    k=4                  # Request 4 final documents
)

print(f"\nQuery: {sample_query_rag}")
print(f"Answer: {final_rag_answer_full}")


Query: What are the main characteristics of a black hole and what is photosynthesis?
Answer: **Black hole**  
- A region of spacetime with gravity so strong that nothing, not even light, can escape once it crosses the event horizon.  
- Forms when a massive star collapses under its own gravity after a supernova.  
- Contains a central singularity where density becomes theoretically infinite.  
- Emits Hawking radiation, which causes it to lose mass and energy over enormous timescales (the rate is inversely proportional to its mass).

**Photosynthesis**  
- The process by which plants convert light energy into chemical energy.  
- Occurs mainly in chloroplasts, specifically within thylakoid membranes.  
- Chlorophyll absorbs blue and red wavelengths and reflects green.  
- The overall reaction uses sunlight to convert carbon dioxide and water into glucose and oxygen.
